w# Credit Line Review Assistant — SCS 3253 Term Project

Report outline this notebook mirrors: Objective -> Data Preparation -> Model Design -> Model Evaluation -> Conclusions.
See `docs/credit-line-review-assistant-proposal.md` for the full design.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import plotly.express as px

from credit_line_review.data import load_cards, load_transactions, load_users

## 1. Objective

We predict a data-driven recommended credit limit for **existing** cardholders from their demographics and observed spending behavior, then flag accounts whose actual limit diverges sharply from that recommendation. This is a retrospective credit-line-review tool, not new-account underwriting.

## 2. Data Preparation

### 2.1 Load raw data

In [2]:
users = load_users()
cards = load_cards()
transactions = load_transactions(nrows=5000)

print("users:", users.shape)
print("cards:", cards.shape)
print("transactions (sample):", transactions.shape)

users: (2000, 14)
cards: (6146, 13)
transactions (sample): (5000, 12)


### 2.2 Does `card_type` affect whether `credit_limit` is meaningful?

A debit card shouldn't carry a real credit limit as a business concept. Check the split before deciding whether to filter.

In [3]:
print(cards["card_type"].value_counts())
print(cards.groupby("card_type")["credit_limit"].describe())

card_type
Debit              3511
Credit             2057
Debit (Prepaid)     578
Name: count, dtype: int64
                  count          mean           std  min      25%      50%  \
card_type                                                                    
Credit           2057.0  11174.380165   6834.713404  0.0   7200.0  10100.0   
Debit            3511.0  18557.888636  12966.142501  0.0  11010.0  16454.0   
Debit (Prepaid)   578.0     64.448097     24.687894  0.0     51.0     65.0   

                     75%       max  
card_type                           
Credit           13900.0   98100.0  
Debit            23478.0  151223.0  
Debit (Prepaid)     80.0     145.0  


If `credit_limit` is populated for Debit cards too, that confirms a data-quality quirk: restrict modeling to `card_type == "Credit"` so the regression target only reflects an actual credit product.

In [4]:
credit_cards = cards[cards["card_type"] == "Credit"]
print("credit cards:", credit_cards.shape, "of", cards.shape[0], "total cards")

credit cards: (2057, 13) of 6146 total cards


### 2.3 Distribution of the regression target and key predictors

Financial quantities (income, debt, credit limit) are usually right-skewed / log-normal. Check skew before deciding whether to log-transform for the linear model.

In [5]:
monetary_cols = {
    "credit_limit": credit_cards["credit_limit"],
    "yearly_income": users["yearly_income"],
    "total_debt": users["total_debt"],
}
for name, series in monetary_cols.items():
    print(f"{name}: skew={series.skew():.2f}, mean={series.mean():.0f}, median={series.median():.0f}")
    fig = px.histogram(series, nbins=50, title=f"Distribution of {name}")
    fig.show()

credit_limit: skew=3.20, mean=11174, median=10100


yearly_income: skew=3.45, mean=45716, median=40744


total_debt: skew=1.81, mean=63710, median=58251


In [6]:
log_limit = np.log1p(credit_cards["credit_limit"])
print(f"log1p(credit_limit): skew={log_limit.skew():.2f}")
fig = px.histogram(log_limit, nbins=50, title="Distribution of log1p(credit_limit)")
fig.show()

log1p(credit_limit): skew=-5.52


A skew close to 0 after `log1p` (vs. the raw skew printed above) would justify log-transforming `credit_limit`, `yearly_income`, and `total_debt` before fitting the linear model — but the log1p skew printed above is large and *negative*, the opposite of what a successful log-transform of a right-skewed variable should look like. Investigate why.

### 2.3.1 Why did `log1p(credit_limit)` get *more* skewed, not less?

A cluster of `credit_limit == 0` rows would sit far below the rest of the (log-transformed) distribution, dragging the skew negative. Check for that directly.

In [7]:
n_zero = (credit_cards["credit_limit"] == 0).sum()
print(f"Credit-card rows with credit_limit == 0: {n_zero} "
      f"({n_zero / len(credit_cards):.1%} of {len(credit_cards)} credit cards)")

Credit-card rows with credit_limit == 0: 26 (1.3% of 2057 credit cards)


These are almost certainly closed/frozen accounts, not a real "what should their limit be" case — a genuine credit-line-review candidate has a nonzero limit today. Rather than silently dropping or blindly modeling them, `build_modeling_table` (Task 4) tags them with an `is_zero_limit` flag: they stay in the exported table for reporting, but are excluded from the train/test split used to fit the regressions, and get their own "closed accounts" segment in the dashboard instead of an under/over-limited flag.

In [8]:
print("credit_score:", users["credit_score"].describe())
fig = px.histogram(users["credit_score"], nbins=40, title="Distribution of credit_score")
fig.show()

fig = px.histogram(transactions["amount"], nbins=50, title="Distribution of transaction amount (sample)")
fig.show()

credit_score: count    2000.000000
mean      709.734500
std        67.221949
min       480.000000
25%       681.000000
50%       711.500000
75%       753.000000
max       850.000000
Name: credit_score, dtype: float64


### 2.4 Feature engineering & modeling table

Aggregate transactions to one row per card, join to cards + users, log-transform monetary columns, and split by `client_id` so a client's cards never span train and test — implemented in `credit_line_review.features` (Tasks 3-4 of the implementation plan) and wired into this notebook in Task 10.

## 3. Model Design

### 3.1 Baseline (predict the training mean)
Not a course model - the floor any real model has to beat.

### 3.2 Linear/Ridge Regression
*Course: Module 5 - Training Models & Feature Selection.* Chosen for coefficient-level interpretability, which matters for a credit decision.

### 3.3 Random Forest Regression
*Course: Module 7 - Decision Trees & Ensemble Learning.* Chosen over SVR (Module 6) - handles non-linear feature interactions without kernel tuning and scales better to this row count.

### 3.4 KMeans persona clustering + PCA visualization
*Course: Module 4 - Clustering (KMeans) and Module 8 - Dimensionality Reduction (PCA).*

### 3.5 Beyond the course
Residual diagnostics, log-transforming skewed monetary columns, grouped train/test splitting, RFM-style behavioral features, and SHAP feature importance are not course topics - see `docs/credit-line-review-assistant-proposal.md` §5.2 for why each one is needed here.

In [9]:
from credit_line_review.features import (
    aggregate_card_behavior,
    build_modeling_table,
    group_train_test_split,
    select_feature_columns,
)
from credit_line_review.models import MeanBaselineRegressor, fit_linear_regression, fit_random_forest
from credit_line_review.evaluation import (
    bootstrap_r2_confidence_interval,
    grouped_cv_metrics,
    permutation_feature_importance,
    regression_metrics,
    residuals,
    shap_feature_importance,
)
from credit_line_review.clustering import fit_kmeans_personas, project_to_2d
from credit_line_review.export import build_scored_accounts, export_scored_accounts
from credit_line_review.config import REFERENCE_DATE

transactions_full = load_transactions(nrows=200_000)
behavior = aggregate_card_behavior(transactions_full, reference_date=REFERENCE_DATE)
table = build_modeling_table(cards, users, behavior, reference_date=REFERENCE_DATE)

# Exclude closed/zero-limit accounts (Task 2 EDA finding) from model fitting -
# they aren't a real "what should their limit be" case. They're scored
# separately below (Section 4.1) and reported as their own segment, not
# dropped from the notebook's output entirely.
modeling_table = table[~table["is_zero_limit"]].reset_index(drop=True)
closed_accounts = table[table["is_zero_limit"]].reset_index(drop=True)
print("modeling rows:", len(modeling_table), "closed/zero-limit rows:", len(closed_accounts))

train, test = group_train_test_split(modeling_table)

# select_feature_columns explicitly excludes identifiers, security fields
# (card_number, cvv, expires, year_pin_last_changed), and raw geographic
# coordinates (latitude/longitude - a fair-lending/redlining risk if used
# directly as a credit-limit predictor) instead of implicitly grabbing every
# numeric/bool column that isn't in a short exclusion list.
feature_cols = select_feature_columns(table)
X_train, y_train = train[feature_cols].fillna(0), train["log_credit_limit"]
X_test, y_test = test[feature_cols].fillna(0), test["log_credit_limit"]

print("train:", X_train.shape, "test:", X_test.shape)
print("features used:", feature_cols)

modeling rows: 2031 closed/zero-limit rows: 26
train: (1616, 24) test: (415, 24)
features used: ['num_cards_issued', 'tenure_months', 'current_age', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'txn_count', 'avg_amount', 'median_amount', 'distinct_mcc', 'distinct_merchant_state', 'recency_days', 'txn_frequency_per_month', 'online_share', 'error_rate', 'log_yearly_income', 'log_total_debt', 'card_brand_Discover', 'card_brand_Mastercard', 'card_brand_Visa', 'has_chip_YES', 'gender_Male']


In [10]:
models = {
    "baseline": MeanBaselineRegressor().fit(X_train, y_train),
    "linear": fit_linear_regression(X_train, y_train),
    "random_forest": fit_random_forest(X_train, y_train),
}

results = []
for name, model in models.items():
    pred_log = model.predict(X_test)
    pred_dollars_ = np.expm1(pred_log)
    actual_dollars_ = np.expm1(y_test)
    metrics = regression_metrics(actual_dollars_, pred_dollars_)
    metrics["model"] = name
    results.append(metrics)

pd.DataFrame(results).set_index("model")

,r2,mae,rmse,mape
model,,,,
baseline,-0.071127,4050.687114,5772.899072,0.474139
linear,0.405337,2965.613192,4301.390970,0.355472
random_forest,0.401957,2984.850592,4313.597126,0.350615


### 3.6 Does this hold up beyond one split? Grouped k-fold cross-validation

The table above is a single train/test split's point estimate. Grouped 5-fold cross-validation (still grouped by `client_id`, so no client's cards span train and test in any fold) reports a mean +/- standard deviation per metric across folds instead, showing whether Random Forest's advantage over the baseline is a stable pattern or an artifact of one particular split.

In [11]:
cv_model_fns = {
    "baseline": lambda X, y: MeanBaselineRegressor().fit(X, y),
    "linear": fit_linear_regression,
    "random_forest": fit_random_forest,
}

cv_rows = []
for name, model_fn in cv_model_fns.items():
    summary = grouped_cv_metrics(
        modeling_table, feature_cols, "log_credit_limit", "client_id", model_fn, n_splits=5,
    )
    summary["model"] = name
    cv_rows.append(summary)

cv_results = pd.DataFrame(cv_rows).set_index("model")
cv_results[["r2_mean", "r2_std", "mae_mean", "mae_std", "rmse_mean", "rmse_std", "mape_mean", "mape_std"]]

,r2_mean,r2_std,mae_mean,mae_std,rmse_mean,rmse_std,mape_mean,mape_std
model,,,,,,,,
baseline,-0.069020,0.030591,4496.677312,418.477371,6900.501738,1179.866350,0.804910,0.355489
linear,0.316200,0.348272,3234.773731,228.085097,5449.525554,1914.249142,0.491221,0.058542
random_forest,0.442578,0.080722,3226.705962,78.333138,4925.435305,545.443316,0.493181,0.043010


In [12]:
best_model = models["random_forest"]
pred_dollars = np.expm1(best_model.predict(X_test))
actual_dollars = np.expm1(y_test)
resid = residuals(actual_dollars, pred_dollars)

fig = px.scatter(x=actual_dollars, y=pred_dollars, labels={"x": "actual credit_limit", "y": "predicted"})
fig.add_shape(type="line", x0=0, y0=0, x1=actual_dollars.max(), y1=actual_dollars.max())
fig.show()

px.histogram(resid, nbins=50, title="Residual distribution").show()

### 3.7 Is the Random Forest's R² advantage a fluke of one test set?

Bootstrap-resampling the held-out test set (with replacement, 1,000 resamples) gives a confidence interval around R² for both the Random Forest and the baseline, rather than trusting a single point estimate each.

In [13]:
baseline_pred_dollars = np.expm1(models["baseline"].predict(X_test))

rf_ci = bootstrap_r2_confidence_interval(actual_dollars, pred_dollars, n_bootstrap=1000)
baseline_ci = bootstrap_r2_confidence_interval(actual_dollars, baseline_pred_dollars, n_bootstrap=1000)

print("Random Forest R2 95% CI:", rf_ci)
print("Baseline R2 95% CI:      ", baseline_ci)

Random Forest R2 95% CI: {'r2_mean': 0.3991542157322212, 'r2_lower': 0.2855645544259292, 'r2_upper': 0.5005053770875223}
Baseline R2 95% CI:       {'r2_mean': -0.07294136497562147, 'r2_lower': -0.11854574213019609, 'r2_upper': -0.031202595833497482}


In [14]:
_, cluster_labels, silhouette = fit_kmeans_personas(X_test, n_clusters=4)
print("silhouette:", silhouette)

coords = project_to_2d(X_test)
coords["cluster"] = cluster_labels
coords["residual"] = resid
px.scatter(coords, x="pc1", y="pc2", color="cluster", title="Card personas (PCA projection)").show()

silhouette: 0.12647507349054282


## 4. Model Evaluation

*Course: Module 3 - Classification.* The regression-vs-actual residual, thresholded, becomes a binary "flag/don't flag" decision - precision/recall-style thinking from Module 3 applies to that decision even though the underlying models are regression/clustering, not classification.

In [15]:
importance = shap_feature_importance(best_model, X_test)
px.bar(importance.head(10), x="mean_abs_shap", y="feature", orientation="h",
       title="Top 10 features by mean |SHAP value|").show()

### 4.2 Permutation importance — a model-agnostic cross-check on SHAP

SHAP's attribution is specific to how the Random Forest was built internally. Permutation importance answers a different question — how much does R² fall if a feature's values are shuffled — as an independent check that the top-ranked features aren't an artifact of SHAP's particular method.

In [16]:
perm_importance = permutation_feature_importance(best_model, X_test, y_test, n_repeats=10)
px.bar(perm_importance.head(10), x="mean_r2_drop", y="feature", orientation="h",
       title="Top 10 features by permutation importance (mean R2 drop)").show()

### 4.1 Closed / zero-limit accounts

The `is_zero_limit` accounts identified in Section 2.3.1 were excluded from training and evaluation above. They're still scored below (using the fitted Random Forest, for reporting purposes only) and exported with a distinct `closed_account` flag rather than being evaluated as under/over-limited or dropped entirely - `build_scored_accounts` (`credit_line_review.export`) applies that override.

In [17]:
flag_threshold = float(np.std(resid) * 1.5)

X_closed = closed_accounts[feature_cols].fillna(0)
closed_pred_dollars = (
    np.expm1(best_model.predict(X_closed)) if len(closed_accounts) else np.array([])
)

export_table = pd.concat([test, closed_accounts], ignore_index=True)
combined_pred_dollars = np.concatenate([pred_dollars, closed_pred_dollars])
# Closed accounts weren't part of the persona clustering above (they were
# excluded from X_test) - sentinel cluster -1 keeps them visually distinct
# in the dashboard's Segment view instead of forcing them into a behavioral
# persona that doesn't apply to a closed account.
combined_cluster_labels = np.concatenate([cluster_labels, np.full(len(closed_accounts), -1)])

scored = build_scored_accounts(
    export_table,
    predicted_limit=combined_pred_dollars,
    cluster_labels=combined_cluster_labels,
    flag_threshold=flag_threshold,
)
export_scored_accounts(scored)
print("exported", len(scored), "rows —", int(scored['is_zero_limit'].sum()), "closed accounts,",
      int((scored['flag'] == 'closed_account').sum()), "flagged closed_account")
scored["flag"].value_counts()

exported 441 rows — 26 closed accounts, 26 flagged closed_account


flag
in_range          363
overlimited        38
closed_account     26
underlimited       14
Name: count, dtype: int64

## 5. Conclusions